# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# Imports & Setup
import re
import json as _json
import math
import logging
import datetime
import urllib.request
import urllib.parse
import urllib.error


In [2]:
import re
import math

# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        # Allow only safe math characters
        safe_expr = re.sub(r"[^0-9+\-*/.() %^]", "", expression).strip()
        if not safe_expr:
            return "Error: empty expression"
        return str(eval(safe_expr, {"__builtins__": {}}, {"math": math}))
    except Exception as e:
        return f"Error in calculation: {e}"


In [3]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text (words longer than 4 chars, top 5 unique)."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []


In [4]:
import re
import json as _json
import urllib.request
import urllib.parse
import urllib.error

# TOOL 3: Wikipedia Fetcher
# Uses the Wikipedia REST summary API.
# Falls back gracefully when there is no internet access.

def wikipedia_fetcher(topic: str) -> str:
    """
    Fetch a short summary for *topic* from Wikipedia.

    Strategy
    --------
    1. Try the Wikipedia REST API  (fast, no extra deps)
    2. On any network error return a clear offline message
    """
    if not topic.strip():
        return "Error: no topic provided for Wikipedia search."

    try:
        # Wikipedia REST summary endpoint
        slug = urllib.parse.quote(topic.strip().replace(" ", "_"))
        url  = f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}"
        req  = urllib.request.Request(
            url,
            headers={"User-Agent": "SingleAgentBot/1.0 (educational project)"}
        )
        with urllib.request.urlopen(req, timeout=6) as resp:
            data = _json.loads(resp.read().decode())
            extract = data.get("extract", "").strip()
            title   = data.get("title", topic)
            if not extract:
                return f"Wikipedia found '{title}' but returned no summary."
            # Return first 3 sentences max
            sentences = re.split(r"(?<=[.!?])\s+", extract)
            return f"[Wikipedia — {title}]\n" + " ".join(sentences[:3])

    except urllib.error.HTTPError as e:
        if e.code == 404:
            return f"Wikipedia: no page found for '{topic}'. Try a different spelling."
        return f"Wikipedia HTTP error {e.code}: {e.reason}"
    except Exception as e:
        # Network blocked / offline — return informative fallback
        return (
            f"[Wikipedia Offline Fallback]\n"
            f"Could not reach Wikipedia for '{topic}' ({type(e).__name__}).\n"
            f"In a live environment this tool fetches the real Wikipedia summary.\n"
            f"Tip: try running this notebook in Google Colab with internet enabled."
        )


In [5]:
import re
import json as _json
import urllib.request

# TOOL 4: Currency Converter
# Uses the Frankfurter public API (no key required).
# Falls back to a hardcoded approximate rate table when offline.

# Approximate fallback rates relative to USD (updated manually)
_FALLBACK_RATES_USD = {
    "USD": 1.0,
    "EUR": 0.92,
    "GBP": 0.79,
    "JPY": 149.50,
    "INR": 83.10,
    "CAD": 1.36,
    "AUD": 1.53,
    "CHF": 0.90,
    "CNY": 7.24,
    "MXN": 17.15,
    "BRL": 4.97,
    "KRW": 1325.0,
    "SGD": 1.34,
    "HKD": 7.82,
    "NOK": 10.55,
    "SEK": 10.40,
    "DKK": 6.89,
    "NZD": 1.63,
    "ZAR": 18.63,
    "RUB": 89.50,
}

def currency_converter(amount: float, from_cur: str, to_cur: str) -> str:
    """
    Convert *amount* from *from_cur* to *to_cur*.

    Strategy
    --------
    1. Try Frankfurter API  (live rates, no API key needed)
    2. Fall back to built-in approximate rate table
    """
    from_cur = from_cur.upper().strip()
    to_cur   = to_cur.upper().strip()

    if from_cur not in _FALLBACK_RATES_USD and to_cur not in _FALLBACK_RATES_USD:
        return f"Error: unknown currency pair '{from_cur}' / '{to_cur}'."

    # Attempt live API
    try:
        url = (
            f"https://api.frankfurter.app/latest"
            f"?amount={amount}&from={from_cur}&to={to_cur}"
        )
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "SingleAgentBot/1.0 (educational project)"}
        )
        with urllib.request.urlopen(req, timeout=6) as resp:
            data = _json.loads(resp.read().decode())
            converted = data["rates"].get(to_cur)
            if converted is None:
                raise ValueError("Currency not in response")
            return (
                f"{amount} {from_cur} = {converted:.4f} {to_cur} "
                f"(live rate via Frankfurter)"
            )
    except Exception:
        pass  # fall through to offline table

    # Offline fallback
    if from_cur not in _FALLBACK_RATES_USD:
        return f"Error: '{from_cur}' not in offline rate table."
    if to_cur not in _FALLBACK_RATES_USD:
        return f"Error: '{to_cur}' not in offline rate table."

    rate = _FALLBACK_RATES_USD[to_cur] / _FALLBACK_RATES_USD[from_cur]
    converted = amount * rate
    return (
        f"{amount} {from_cur} ≈ {converted:.4f} {to_cur} "
        f"(approximate offline rate; live rate unavailable)"
    )


In [6]:
import logging
import datetime

# Logger Setup
# Writes every agent call to agent.log (append mode) AND streams to console.

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# File handler - creates / appends to agent.log in the working directory
_file_handler = logging.FileHandler("agent.log", mode="a", encoding="utf-8")
_file_handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
))

logger = logging.getLogger("agent")
logger.addHandler(_file_handler)
logger.setLevel(logging.INFO)

def _log(query: str, route: str, result) -> None:
    """Emit one structured log line per agent call."""
    truncated = str(result)[:120] + ("…" if len(str(result)) > 120 else "")
    logger.info(f'ROUTE={route:<12} | QUERY={repr(query)[:80]:<82} | RESULT={truncated}')


In [7]:
import re

# Confidence Scorer
# Scores each route by counting weighted keyword/pattern hits in the query.
# The route with the highest score wins; ties fall to "general".

_ROUTE_SIGNALS: dict[str, list[tuple[str, float]]] = {
    "calculation": [
        (r"\bcalculate\b",           3.0),
        (r"\bcompute\b",             2.5),
        (r"\beval(uate)?\b",         2.0),
        (r"\bsolve\b",               1.5),
        (r"\bwhat(?:'s| is)\s+\d",  1.5),
        (r"[+\-*/%^]",                1.0),   # operator present
        (r"\d+\s*[+\-*/]\s*\d+",  2.0),   # explicit arithmetic
    ],
    "keywords": [
        (r"\bkeywords?\b",           3.0),
        (r"\bextract\b",             2.0),
        (r"\bkey\s*phrase",          2.0),
        (r"\btag\b",                 1.0),
        (r"\bimportant\s+words\b",  1.5),
    ],
    "wikipedia": [
        (r"\bwikipedia\b",           3.5),
        (r"\bwho\s+is\b",           2.0),
        (r"\bwhat\s+is\b",          1.5),
        (r"\btell\s+me\s+about\b", 2.0),
        (r"\bhistory\s+of\b",       1.5),
        (r"\bdefinition\s+of\b",    1.5),
        (r"\bexplain\b",             1.0),
    ],
    "currency": [
        (r"\bcurrenc(y|ies)\b",      3.0),
        (r"\bconvert\b",             2.5),
        (r"\bexchange\b",            2.5),
        (r"\b(usd|eur|gbp|inr|jpy|cad|aud|chf|cny)\b", 2.0),
        (r"\bhow\s+much\s+is\b",   1.5),
        (r"\brate\b",                1.0),
    ],
}

def confidence_score(query: str) -> dict[str, float]:
    """
    Return a {route: score} dict for *query*.
    Scores are non-negative; 0 means no evidence for that route.
    """
    q = query.lower()
    scores: dict[str, float] = {}
    for route, signals in _ROUTE_SIGNALS.items():
        total = 0.0
        for pattern, weight in signals:
            if re.search(pattern, q):
                total += weight
        scores[route] = round(total, 2)
    return scores

def best_route(query: str) -> str:
    """Return the highest-scoring route name, or 'general' on a tie/zero."""
    scores = confidence_score(query)
    top_score = max(scores.values())
    if top_score == 0:
        return "general"
    # Pick the route with the max score (stable: first occurrence wins ties)
    return max(scores, key=lambda r: scores[r])


In [ ]:
import re

# AGENT FUNCTION  (confidence-scored routing + logging)

def _parse_currency_query(query: str):
    """
    Extract (amount, from_currency, to_currency) from natural-language query.
    Returns None if parsing fails.

    Handles patterns like:
      - "convert 100 USD to EUR"
      - "100 dollars in euros"
      - "exchange 50 GBP to INR"
      - "how much is 200 USD in JPY"
    """
    q = query.upper()

    # Pattern A: "convert/exchange X CUR1 to CUR2"
    m = re.search(
        r"(?:CONVERT|EXCHANGE|HOW\s+MUCH\s+IS)?\s*([\d,.]+)\s*([A-Z]{3})\s*(?:TO|IN)\s*([A-Z]{3})",
        q
    )
    if m:
        try:
            amount = float(m.group(1).replace(",", ""))
            return amount, m.group(2), m.group(3)
        except ValueError:
            pass

    # Pattern B: list of known currencies in order
    cur_pattern = r"\b(USD|EUR|GBP|JPY|INR|CAD|AUD|CHF|CNY|MXN|BRL|KRW|SGD|HKD|NOK|SEK|DKK|NZD|ZAR|RUB)\b"
    currencies = re.findall(cur_pattern, q)
    amounts    = re.findall(r"[\d,.]+", q)
    if len(currencies) >= 2 and amounts:
        try:
            amount = float(amounts[0].replace(",", ""))
            return amount, currencies[0], currencies[1]
        except ValueError:
            pass

    return None   # couldn't parse


def agent(query: str) -> dict:
    """
    Single-agent smart assistant.

    Routing (confidence-scored)
    ---------------------------
    calculation  →  Calculator Tool
    keywords     →  Keyword Extractor Tool
    wikipedia    →  Wikipedia Fetcher Tool
    currency     →  Currency Converter Tool
    general      →  Direct text response
    """
    if not isinstance(query, str) or not query.strip():
        result = {"type": "error", "result": "Empty or invalid query."}
        _log(str(query), "error", result["result"])
        return result

    route  = best_route(query)
    scores = confidence_score(query)

    # calculation
    if route == "calculation":
        expr = re.sub(r"(?:calculate|compute|evaluate|solve)\s*", "", query,
                      flags=re.IGNORECASE).strip()
        expr = re.sub(r"[^0-9+\-*/.() %^]", "", expr).strip()
        if not expr:
            result = {"type": "error", "result": "No valid math expression found."}
        else:
            result = {"type": "calculation", "result": calculator(expr),
                      "confidence_scores": scores}

    # keywords
    elif route == "keywords":
        text = re.sub(r"(extract\s+)?keywords?\s*(from|in|of)?\s*", "", query,
                      flags=re.IGNORECASE).strip()
        if not text:
            result = {"type": "error", "result": "No text provided for keyword extraction."}
        else:
            result = {"type": "keywords", "result": extract_keywords(text),
                      "confidence_scores": scores}

    # wikipedia
    elif route == "wikipedia":
        topic = re.sub(
            r"(wikipedia\s*)?(what\s+is|who\s+is|tell\s+me\s+about|"
            r"history\s+of|definition\s+of|explain)\s*",
            "", query, flags=re.IGNORECASE
        ).strip(" ?")
        result = {"type": "wikipedia", "result": wikipedia_fetcher(topic),
                  "confidence_scores": scores}

    # currency
    elif route == "currency":
        parsed = _parse_currency_query(query)
        if parsed is None:
            result = {
                "type": "error",
                "result": (
                    "Could not parse currency query. "
                    "Try: 'convert 100 USD to EUR'"
                )
            }
        else:
            amount, from_cur, to_cur = parsed
            result = {
                "type": "currency",
                "result": currency_converter(amount, from_cur, to_cur),
                "confidence_scores": scores
            }

    # general fallback
    else:
        result = {
            "type": "general",
            "result": (
                f"I received: \"{query}\". "
                "I can help with: calculations, keyword extraction, "
                "Wikipedia lookups, and currency conversions."
            ),
            "confidence_scores": scores
        }

    _log(query, route, result.get("result", ""))
    return result


## 📦 Output Format

```json
{
  "type": "calculation | keywords | wikipedia | currency | general | error",
  "result": "...",
  "confidence_scores": {"calculation": 3.0, "keywords": 0.0, "wikipedia": 1.5, "currency": 2.5}
}
```


In [9]:
# 🧪 Test Cases — All Tools

import pprint
pp = pprint.PrettyPrinter(width=80, sort_dicts=False)

test_suite = [
    # Original tests
    ("Calculate 20 + 5",                                       "calculation"),
    ("Extract keywords from Artificial Intelligence is transforming industries",
                                                               "keywords"),
    ("What is machine learning?",                              "wikipedia"),

    # Calculator
    ("Calculate (100 * 3.14) / 2",                            "calculation"),
    ("Compute 2 ** 10",                                        "calculation"),
    ("Evaluate 144 / 12 + 7",                                  "calculation"),

    # Keyword Extractor
    ("Extract keywords from Deep learning is revolutionizing NLP and computer vision",
                                                               "keywords"),
    ("What are the keywords in: neural networks enable pattern recognition?",
                                                               "keywords"),

    # Wikipedia Fetcher
    ("Tell me about Jensen Huang?",                          "wikipedia"),
    ("Who is Sonam Wangchuk?",                                   "wikipedia"),
    ("What is the transformer architecture?",                             "wikipedia"),
    ("Wikipedia: Indian Kingdom",                 "wikipedia"),

    # Currency Converter
    ("Convert 100 USD to INR",                                 "currency"),
    ("How much is 500 INR in USD?",                            "currency"),
    ("Exchange 250 GBP to JPY",                                "currency"),
    ("convert 1000 EUR to INR",                                "currency"),

    # Edge / Error cases
    ("Calculate",                                              "error"),
    ("Extract keywords from",                                  "error"),
    ("",                                                       "error"),
]

PASS = FAIL = 0
print("  SINGLE-AGENT PIPELINE")

for query, expected_type in test_suite:
    resp = agent(query)
    actual = resp.get("type")
    status = "PASS" if actual == expected_type else f"FAIL (got '{actual}')"
    if actual == expected_type:
        PASS += 1
    else:
        FAIL += 1

    print(f"\n{status} | expected={expected_type}")
    print(f"  Query : {repr(query)[:70]}")
    print(f"  Result: {str(resp.get('result',''))[:100]}")
    if "confidence_scores" in resp:
        scores = resp["confidence_scores"]
        top = sorted(scores.items(), key=lambda x: -x[1])[:3]
        print(f"  Top scores: {top}")

print(f"  Results: {PASS} passed, {FAIL} failed out of {PASS+FAIL} tests")


2026-06-28 00:52:45 | INFO | ROUTE=calculation  | QUERY='Calculate 20 + 5'                                                                 | RESULT=25
2026-06-28 00:52:45 | INFO | ROUTE=keywords     | QUERY='Extract keywords from Artificial Intelligence is transforming industries'         | RESULT=['transforming', 'intelligence', 'industries', 'artificial']


  SINGLE-AGENT PIPELINE

PASS | expected=calculation
  Query : 'Calculate 20 + 5'
  Result: 25
  Top scores: [('calculation', 6.0), ('keywords', 0.0), ('wikipedia', 0.0)]

PASS | expected=keywords
  Query : 'Extract keywords from Artificial Intelligence is transforming industr
  Result: ['transforming', 'intelligence', 'industries', 'artificial']
  Top scores: [('keywords', 5.0), ('calculation', 0.0), ('wikipedia', 0.0)]


2026-06-28 00:52:45 | INFO | ROUTE=wikipedia    | QUERY='What is machine learning?'                                                        | RESULT=[Wikipedia — Machine learning]
Machine learning (ML) is a field of study in artificial intelligence concerned with the d…
2026-06-28 00:52:45 | INFO | ROUTE=calculation  | QUERY='Calculate (100 * 3.14) / 2'                                                       | RESULT=157.0
2026-06-28 00:52:45 | INFO | ROUTE=calculation  | QUERY='Compute 2 ** 10'                                                                  | RESULT=1024
2026-06-28 00:52:45 | INFO | ROUTE=calculation  | QUERY='Evaluate 144 / 12 + 7'                                                            | RESULT=19.0
2026-06-28 00:52:45 | INFO | ROUTE=keywords     | QUERY='Extract keywords from Deep learning is revolutionizing NLP and computer vision'   | RESULT=['learning', 'vision', 'computer', 'revolutionizing']
2026-06-28 00:52:45 | INFO | ROUTE=keywords     | QUERY='What are th


PASS | expected=wikipedia
  Query : 'What is machine learning?'
  Result: [Wikipedia — Machine learning]
Machine learning (ML) is a field of study in artificial intelligence 
  Top scores: [('wikipedia', 1.5), ('calculation', 0.0), ('keywords', 0.0)]

PASS | expected=calculation
  Query : 'Calculate (100 * 3.14) / 2'
  Result: 157.0
  Top scores: [('calculation', 6.0), ('keywords', 0.0), ('wikipedia', 0.0)]

PASS | expected=calculation
  Query : 'Compute 2 ** 10'
  Result: 1024
  Top scores: [('calculation', 3.5), ('keywords', 0.0), ('wikipedia', 0.0)]

PASS | expected=calculation
  Query : 'Evaluate 144 / 12 + 7'
  Result: 19.0
  Top scores: [('calculation', 5.0), ('keywords', 0.0), ('wikipedia', 0.0)]

PASS | expected=keywords
  Query : 'Extract keywords from Deep learning is revolutionizing NLP and comput
  Result: ['learning', 'vision', 'computer', 'revolutionizing']
  Top scores: [('keywords', 5.0), ('calculation', 0.0), ('wikipedia', 0.0)]

PASS | expected=keywords
  Query : 'Wh

2026-06-28 00:52:45 | INFO | ROUTE=wikipedia    | QUERY='Tell me about Jensen Huang?'                                                      | RESULT=[Wikipedia — Jensen Huang]
Jen-Hsun "Jensen" Huang is a Taiwanese and American business executive and electrical enginee…


...


PASS | expected=currency
  Query : 'How much is 500 INR in USD?'
  Result: 500.0 INR = 5.2964 USD (live rate via Frankfurter)
  Top scores: [('currency', 3.5), ('calculation', 0.0), ('keywords', 0.0)]


2026-06-28 00:52:49 | INFO | ROUTE=currency     | QUERY='Exchange 250 GBP to JPY'                                                          | RESULT=250.0 GBP = 53419.0000 JPY (live rate via Frankfurter)



PASS | expected=currency
  Query : 'Exchange 250 GBP to JPY'
  Result: 250.0 GBP = 53419.0000 JPY (live rate via Frankfurter)
  Top scores: [('currency', 4.5), ('calculation', 0.0), ('keywords', 0.0)]


2026-06-28 00:52:50 | INFO | ROUTE=currency     | QUERY='convert 1000 EUR to INR'                                                          | RESULT=1000.0 EUR = 107631.0000 INR (live rate via Frankfurter)
2026-06-28 00:52:50 | INFO | ROUTE=calculation  | QUERY='Calculate'                                                                        | RESULT=No valid math expression found.
2026-06-28 00:52:50 | INFO | ROUTE=keywords     | QUERY='Extract keywords from'                                                            | RESULT=No text provided for keyword extraction.
2026-06-28 00:52:50 | INFO | ROUTE=error        | QUERY=''                                                                                 | RESULT=Empty or invalid query.



PASS | expected=currency
  Query : 'convert 1000 EUR to INR'
  Result: 1000.0 EUR = 107631.0000 INR (live rate via Frankfurter)
  Top scores: [('currency', 4.5), ('calculation', 0.0), ('keywords', 0.0)]

PASS | expected=error
  Query : 'Calculate'
  Result: No valid math expression found.

PASS | expected=error
  Query : 'Extract keywords from'
  Result: No text provided for keyword extraction.

PASS | expected=error
  Query : ''
  Result: Empty or invalid query.
  Results: 19 passed, 0 failed out of 19 tests


In [10]:
# 🎯 Interactive Mode
# Run in a live Jupyter / Colab session. Type 'exit' to stop.
# Type 'scores <query>' to see confidence scores without routing.

while True:
    user_input = input("\nEnter query (or 'exit'): ").strip()
    if not user_input:
        continue
    if user_input.lower() == "exit":
        print("Session ended.")
        break
    if user_input.lower().startswith("scores "):
        q = user_input[7:]
        print("Confidence scores:", confidence_score(q))
        print("Best route:", best_route(q))
        continue
    result = agent(user_input)
    import pprint
    pprint.pprint(result)


2026-06-28 00:53:00 | INFO | ROUTE=wikipedia    | QUERY='What is GTA VI?'                                                                  | RESULT=[Wikipedia — Grand Theft Auto VI]
Grand Theft Auto VI is an upcoming action-adventure game developed and published by Ro…


{'confidence_scores': {'calculation': 0.0,
                       'currency': 0.0,
                       'keywords': 0.0,
                       'wikipedia': 1.5},
 'result': '[Wikipedia — Grand Theft Auto VI]\n'
           'Grand Theft Auto VI is an upcoming action-adventure game developed '
           'and published by Rockstar Games. It is due to be the eighth main '
           'Grand Theft Auto game, following Grand Theft Auto V (2013), and '
           'the sixteenth entry overall. Set within the fictional US state of '
           'Leonida, based on Florida, the story follows the romantic criminal '
           'duo of Jason Duval and Lucia Caminos.',
 'type': 'wikipedia'}


2026-06-28 00:53:12 | INFO | ROUTE=wikipedia    | QUERY='What is Capitol Hill mystery soda machine'                                        | RESULT=[Wikipedia — Capitol Hill mystery soda machine]
The Capitol Hill mystery soda machine was a vending machine in Capitol H…


{'confidence_scores': {'calculation': 0.0,
                       'currency': 0.0,
                       'keywords': 0.0,
                       'wikipedia': 1.5},
 'result': '[Wikipedia — Capitol Hill mystery soda machine]\n'
           'The Capitol Hill mystery soda machine was a vending machine in '
           'Capitol Hill, Seattle, notable for its "mystery" buttons which '
           'dispensed unusual drink flavors. It is unknown who restocked the '
           'machine; this originally caused the development of a local legend '
           'that the machine was haunted, and later an enduring legacy of '
           '"cultural fascination". The machine reportedly operated from the '
           'late 1990s until its unexplained disappearance in 2018.',
 'type': 'wikipedia'}


2026-06-28 00:55:01 | INFO | ROUTE=wikipedia    | QUERY='What is Red Star OS'                                                              | RESULT=[Wikipedia — Red Star OS]
Red Star OS is a North Korean Linux distribution, with development first starting in 1998 at t…


{'confidence_scores': {'calculation': 0.0,
                       'currency': 0.0,
                       'keywords': 0.0,
                       'wikipedia': 1.5},
 'result': '[Wikipedia — Red Star OS]\n'
           'Red Star OS is a North Korean Linux distribution, with development '
           'first starting in 1998 at the Korea Computer Center (KCC).',
 'type': 'wikipedia'}


2026-06-28 00:55:21 | INFO | ROUTE=currency     | QUERY='7499 INR to USD'                                                                  | RESULT=7499.0 INR = 79.4380 USD (live rate via Frankfurter)


{'confidence_scores': {'calculation': 0.0,
                       'currency': 2.0,
                       'keywords': 0.0,
                       'wikipedia': 0.0},
 'result': '7499.0 INR = 79.4380 USD (live rate via Frankfurter)',
 'type': 'currency'}
Session ended.
